# LeetCode 631 — Design Excel Sum Formula bằng Python

- Link: https://leetcode.com/problems/design-excel-sum-formula/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Thiết kế lớp (Design), Đồ thị phụ thuộc (DAG), Đệ quy có nhớ, Sắp xếp tô-pô (topological sort), Phân tích chuỗi

> **Ghi chú Premium:** đây là bài dành cho tài khoản LeetCode Premium. Phần đề bài dưới đây được viết lại theo hiểu biết về bài (chữ ký lớp, ví dụ, ràng buộc). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại** chi tiết đề (đặc biệt: kiểu tham số `width`/`column` là `str` một ký tự, và giới hạn số lời gọi) trước khi nộp.

Skeleton trên LeetCode (Python3):

```python
class Excel:

    def __init__(self, height: int, width: str):

    def set(self, row: int, column: str, val: int) -> None:

    def get(self, row: int, column: str) -> int:

    def sum(self, row: int, column: str, numbers: List[str]) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài này mô phỏng một bảng tính Excel thu nhỏ: ô có thể chứa **số** hoặc **công thức tổng** tham chiếu tới ô khác. Khi một ô thay đổi, mọi ô phụ thuộc vào nó (trực tiếp hay gián tiếp) cũng phải thay đổi theo. Đây là bài **thiết kế** — không có "một công thức thần kỳ", mà là chọn **cách lưu trữ** và **khi nào tính toán** (lười hay háo hức).

Sau notebook này bạn sẽ:

1. Biết phân tích chuỗi tham chiếu `"A1"`, `"B12"`, `"A1:C3"` thành danh sách ô.
2. Mô hình hóa công thức như **đồ thị có hướng không chu trình (DAG)** với cạnh có **bội số** (một ô có thể được cộng nhiều lần).
3. Cài đặt chiến lược **lười** (lazy): chỉ tính khi `get`, có/không có nhớ (memo).
4. Cài đặt chiến lược **háo hức** (eager): cập nhật lan truyền ngay khi `set`, bằng lan truyền chênh lệch (delta) hoặc **sắp xếp tô-pô**.
5. Kiểm chứng 4 cách bằng chuỗi thao tác ngẫu nhiên có seed, đảm bảo không tạo chu trình.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Thiết kế lớp `Excel`:

- `Excel(height, width)`: tạo bảng có các hàng `1..height` và các cột `'A'..width` (`width` là một chữ cái). Mọi ô ban đầu bằng `0`.
- `set(row, column, val)`: đặt ô `(row, column)` bằng số `val`. Nếu ô đang có công thức thì **công thức bị xóa**.
- `get(row, column)`: trả về giá trị hiện tại của ô.
- `sum(row, column, numbers)`: đặt ô `(row, column)` thành **công thức tổng** của các tham chiếu trong `numbers`, và **trả về** giá trị tổng hiện tại. Mỗi phần tử của `numbers` có dạng:
  - `"ColRow"` — một ô, ví dụ `"F7"`;
  - `"ColRow1:ColRow2"` — hình chữ nhật từ góc trên-trái tới góc dưới-phải, ví dụ `"F7:H9"`.

  Công thức **tồn tại lâu dài**: khi các ô được tham chiếu thay đổi, giá trị ô công thức tự cập nhật — cho tới khi ô bị `set` hoặc `sum` mới ghi đè.

Lưu ý quan trọng:
- Nếu một ô xuất hiện **nhiều lần** (ví dụ `["A1", "A1:B2"]` chứa `A1` hai lần) thì nó được cộng **nhiều lần**.
- Đề đảm bảo **không có tham chiếu vòng** (chu trình).

**Ví dụ**

```text
Excel(3, "C")
  bảng 3 hàng x 3 cột (A, B, C), toàn 0
set(1, "A", 2)
  A1 = 2
sum(3, "C", ["A1", "A1:B2"])   -> 4
  C3 = A1 + (A1 + B1 + A2 + B2) = 2 + (2 + 0 + 0 + 0) = 4
set(2, "B", 2)
  B2 = 2  =>  C3 tự cập nhật thành 2 + (2 + 0 + 0 + 2) = 6
get(3, "C")                    -> 6
```

**Ràng buộc (theo hiểu biết về bài)**

- `1 <= height <= 26`, `'A' <= width <= 'Z'`
- `1 <= row <= height`, `'A' <= column <= width`
- `-100 <= val <= 100`
- `1 <= numbers.length <= 5`, mỗi phần tử hợp lệ theo định dạng trên
- Tổng số lời gọi `set`, `get`, `sum` không quá `100`.

In [1]:
# Mô phỏng ví dụ bằng tay với dict để "cảm" bài toán trước khi thiết kế lớp
cells = {}                        # (row, col) -> số
formula = {}                      # (row, col) -> list ô tham chiếu (có thể trùng)

def value(cell):
    if cell in formula:
        return sum(value(x) for x in formula[cell])
    return cells.get(cell, 0)

cells[(1, 'A')] = 2
formula[(3, 'C')] = [(1, 'A'), (1, 'A'), (1, 'B'), (2, 'A'), (2, 'B')]
print("C3 sau sum:", value((3, 'C')))
cells[(2, 'B')] = 2
print("C3 sau set B2 = 2:", value((3, 'C')))

C3 sau sum: 4
C3 sau set B2 = 2: 6


Cell trên đã là một lời giải "lười" thô sơ: công thức lưu danh sách ô, `value` đệ quy. Phần còn lại của notebook làm việc này **có tổ chức**, đúng giao diện LeetCode, và nhanh hơn.

## 3. Kiến thức nền

### 3.1. Chuyển cột chữ ↔ chỉ số: `ord` và `chr`

`ord('A') = 65`. Cột `'C'` có chỉ số `ord('C') - ord('A') = 2`. Ngược lại `chr(65 + 2) = 'C'`.

In [2]:
for ch in "ABCZ":
    print(ch, "->", ord(ch) - ord('A'))
print([chr(ord('A') + i) for i in range(5)])

A -> 0
B -> 1
C -> 2
Z -> 25
['A', 'B', 'C', 'D', 'E']


### 3.2. Phân tích chuỗi tham chiếu: `split`, cắt chuỗi, `int`

`"B12"` → cột `'B'` (ký tự đầu), hàng `int("12") = 12` (phần còn lại). `"A1:C3".split(':')` → `["A1", "C3"]`.

In [3]:
def parse_cell(ref):
    """'B12' -> (12, 'B')"""
    return int(ref[1:]), ref[0]

def expand(ref):
    """'A1' -> [(1,'A')];  'A1:B2' -> [(1,'A'), (1,'B'), (2,'A'), (2,'B')]"""
    if ':' in ref:
        a, b = ref.split(':')
        (r1, c1), (r2, c2) = parse_cell(a), parse_cell(b)
        return [(r, chr(c)) for r in range(r1, r2 + 1) for c in range(ord(c1), ord(c2) + 1)]
    return [parse_cell(ref)]

print(parse_cell("B12"), expand("A1"), expand("A1:B2"))
print(expand("B2:C3"))

(12, 'B') [(1, 'A')] [(1, 'A'), (1, 'B'), (2, 'A'), (2, 'B')]
[(2, 'B'), (2, 'C'), (3, 'B'), (3, 'C')]


### 3.3. `collections.Counter` — đếm bội số tham chiếu

Một công thức có thể chứa cùng một ô nhiều lần. `Counter` gom lại thành `ô → số lần`, giúp tính tổng bằng `Σ value(ô) · số lần`.

In [4]:
from collections import Counter
refs = expand("A1") + expand("A1:B2")
cnt = Counter(refs)
print(cnt)
print("A1 xuất hiện", cnt[(1, 'A')], "lần")

Counter({(1, 'A'): 2, (1, 'B'): 1, (2, 'A'): 1, (2, 'B'): 1})
A1 xuất hiện 2 lần


### 3.4. Đồ thị phụ thuộc và cạnh ngược

Nếu `C3 = A1 + B2` thì `C3` **phụ thuộc** `A1`, `B2`. Ta lưu hai chiều:

- `formula[C3] = {A1: 1, B2: 1}` (C3 cần ai) — dùng khi **tính** C3.
- `dependents[A1] = {C3: 1}` (ai cần A1) — dùng khi A1 **thay đổi** để biết phải cập nhật ai.

In [5]:
from collections import defaultdict
formula_demo = {"C3": Counter({"A1": 2, "B2": 1}), "D1": Counter({"C3": 1})}
dependents_demo = defaultdict(Counter)
for cell, refs in formula_demo.items():
    for ref, k in refs.items():
        dependents_demo[ref][cell] += k
print(dict(dependents_demo))

{'A1': Counter({'C3': 2}), 'B2': Counter({'C3': 1}), 'C3': Counter({'D1': 1})}


### 3.5. Sắp xếp tô-pô (thuật toán Kahn)

Trong DAG, thứ tự tô-pô là thứ tự mà mỗi đỉnh đứng **sau** mọi đỉnh nó phụ thuộc. Kahn: đếm "bậc vào" mỗi đỉnh, lấy dần các đỉnh bậc vào 0.

In [6]:
from collections import deque

def kahn(nodes, edges):
    """edges: list (u, v) nghĩa là u phải đứng trước v."""
    indeg = {x: 0 for x in nodes}
    out = defaultdict(list)
    for u, v in edges:
        out[u].append(v)
        indeg[v] += 1
    q = deque(x for x in nodes if indeg[x] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in out[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order

print(kahn(["A1", "B2", "C3", "D1"], [("A1", "C3"), ("B2", "C3"), ("C3", "D1")]))

['A1', 'B2', 'C3', 'D1']


### 3.6. Thiết kế lớp trong Python

Phương thức có tham số `self` trỏ tới đối tượng; thuộc tính khởi tạo trong `__init__`. Lưu ý: lớp `Excel` có phương thức tên `sum` — **bên trong lớp** gọi hàm `sum` có sẵn của Python vẫn được (vì `self.sum` và `sum` khác nhau), nhưng dễ gây rối khi đọc; ta sẽ viết vòng lặp rõ ràng.

In [7]:
class Counter2:
    def __init__(self):
        self.total = 0
    def sum(self, xs):            # trùng tên hàm có sẵn: OK vì gọi qua self.sum
        self.total = sum(xs)      # 'sum' ở đây là hàm có sẵn
        return self.total

c = Counter2()
print(c.sum([1, 2, 3]), c.total)

6 6


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Hai câu hỏi thiết kế

1. **Lưu gì cho mỗi ô?** Một số (`val`) **hoặc** một công thức (`Counter` các ô tham chiếu).
2. **Khi nào tính giá trị ô công thức?**
   - **Lười (lazy):** chỉ tính khi `get`/`sum` hỏi. `set` rất nhanh, `get` có thể chậm.
   - **Háo hức (eager):** tính ngay khi có thay đổi, lan truyền tới các ô phụ thuộc. `get` là `O(1)`, `set` chậm hơn.

### 4.2. Tính tay một chuỗi phụ thuộc nhiều tầng

```text
Excel(2, "C")
set(1, "A", 5)                 A1 = 5
sum(1, "B", ["A1", "A1"])      B1 = 2·A1 = 10
sum(1, "C", ["A1:B1"])         C1 = A1 + B1 = 15
sum(2, "A", ["B1", "C1"])      A2 = B1 + C1 = 25
set(1, "A", 1)                 A1 = 1  =>  B1 = 2, C1 = 3, A2 = 5
```

Khi đổi `A1`, ảnh hưởng lan theo **nhiều đường**: `A1 → B1 → C1 → A2`, `A1 → C1 → A2`, `A1 → B1 → A2`. Với cách háo hức, ta phải cập nhật đúng **mỗi ô một lần với giá trị cuối cùng** (hoặc cộng dồn chênh lệch theo mọi đường cho đúng).

### 4.3. Ghi đè công thức

```text
sum(1, "B", ["A1"])     B1 = A1
set(1, "B", 7)          B1 = 7 — công thức cũ biến mất, đổi A1 không còn ảnh hưởng B1
```

Với cách háo hức, phải **xóa cạnh ngược** của công thức cũ, nếu không B1 vẫn bị cập nhật nhầm.

In [8]:
cells, formula = {}, {}
cells[(1, 'A')] = 5
formula[(1, 'B')] = expand("A1") * 2
formula[(1, 'C')] = expand("A1:B1")
formula[(2, 'A')] = expand("B1") + expand("C1")
print("trước:", {k: value(k) for k in [(1, 'A'), (1, 'B'), (1, 'C'), (2, 'A')]})
cells[(1, 'A')] = 1
print("sau  :", {k: value(k) for k in [(1, 'A'), (1, 'B'), (1, 'C'), (2, 'A')]})

trước: {(1, 'A'): 5, (1, 'B'): 10, (1, 'C'): 15, (2, 'A'): 25}
sau  : {(1, 'A'): 1, (1, 'B'): 2, (1, 'C'): 3, (2, 'A'): 5}


## 5. Cách 1 — Brute Force: lười, đệ quy không nhớ

**Ý tưởng.** Lưu `vals` (số) và `formulas` (`Counter`). `get` đệ quy: ô có công thức thì cộng `get(ref) · k` cho mọi tham chiếu; không thì trả `vals`. `set` xóa công thức và đặt số. `sum` lưu công thức rồi trả `get`.

In [9]:
from typing import List

class ExcelLazy:
    def __init__(self, height: int, width: str):
        self.vals = [[0] * (ord(width) - 64) for _ in range(height + 1)]   # hàng 1..height
        self.formulas = {}                                                  # (r, c) -> Counter

    @staticmethod
    def _cells(numbers):
        cnt = Counter()
        for ref in numbers:
            for r, col in expand(ref):
                cnt[(r, ord(col) - 65)] += 1
        return cnt

    def _value(self, r, c):
        f = self.formulas.get((r, c))
        if f is None:
            return self.vals[r][c]
        total = 0
        for (rr, cc), k in f.items():
            total += self._value(rr, cc) * k
        return total

    def set(self, row: int, column: str, val: int) -> None:
        c = ord(column) - 65
        self.formulas.pop((row, c), None)
        self.vals[row][c] = val

    def get(self, row: int, column: str) -> int:
        return self._value(row, ord(column) - 65)

    def sum(self, row: int, column: str, numbers: List[str]) -> int:
        self.formulas[(row, ord(column) - 65)] = self._cells(numbers)
        return self.get(row, column)

ex = ExcelLazy(3, "C")
ex.set(1, "A", 2)
r1 = ex.sum(3, "C", ["A1", "A1:B2"])
ex.set(2, "B", 2)
r2 = ex.get(3, "C")
print(r1, r2)
assert (r1, r2) == (4, 6)

4 6


**Đọc code:**

- `vals` có `height + 1` hàng để dùng trực tiếp chỉ số hàng `1..height` (bỏ hàng 0).
- `ord(width) - 64` = số cột (`'A'` → 1).
- `_cells` biến `numbers` thành `Counter` các ô `(hàng, chỉ số cột)`.
- `_value` đệ quy theo công thức; không có chu trình nên đệ quy dừng.

**Độ phức tạp.** `set`: `O(1)`. `sum`: `O(số ô trong các vùng)` để dựng `Counter` + một `get`. `get`: **có thể hàm mũ** theo độ sâu phụ thuộc — vì không nhớ, một ô bị tính lại theo **mỗi đường** dẫn tới nó.

**Nút thắt.** Ví dụ "chuỗi kim cương": `X2 = X1 + X1`, `X3 = X2 + X2`, ... (dùng hai tham chiếu riêng rẽ để `Counter` vẫn gộp — nhưng nếu tham chiếu qua hai ô trung gian khác nhau thì không gộp được). Cell dưới dựng chuỗi mà mỗi tầng tham chiếu **hai ô** của tầng trước, đếm số lần gọi `_value`.

In [10]:
import time

def diamond(excel_cls, depth):
    """Hàng r có 2 ô A_r, B_r; mỗi ô ở hàng r = A_{r-1} + B_{r-1}. Số đường đi tăng 2^depth."""
    ex = excel_cls(depth, "B")
    ex.set(1, "A", 1)
    ex.set(1, "B", 1)
    for r in range(2, depth + 1):
        ex.sum(r, "A", [f"A{r - 1}", f"B{r - 1}"])
        ex.sum(r, "B", [f"A{r - 1}:B{r - 1}"])
    return ex

calls = [0]
orig = ExcelLazy._value
def counting_value(self, r, c):
    calls[0] += 1
    return orig(self, r, c)
ExcelLazy._value = counting_value
for depth in (8, 12, 16):
    calls[0] = 0
    ex = diamond(ExcelLazy, depth)
    calls[0] = 0
    t = time.perf_counter()
    v = ex.get(depth, "A")
    print(f"depth={depth}: get = {v}, số lần gọi _value = {calls[0]}, {time.perf_counter() - t:.4f}s")
ExcelLazy._value = orig

depth=8: get = 128, số lần gọi _value = 255, 0.0000s
depth=12: get = 2048, số lần gọi _value = 4095, 0.0006s
depth=16: get = 32768, số lần gọi _value = 65535, 0.0101s


Số lần gọi tăng gấp đôi mỗi tầng → `2^depth`. Với 26 hàng là ~`10^8` — quá chậm. Đó là động lực cho các cách sau.

## 6. Bước đột phá

**Insight 1 (DAG có trọng số).** Công thức tạo thành một **DAG**: cạnh `ref → cell` với trọng số = số lần `ref` xuất hiện trong công thức của `cell`. Giá trị ô là tổ hợp tuyến tính của các ô "lá" (ô số).

**Insight 2 (nhớ khi tính lười).** Trong **một** lần `get`, mỗi ô chỉ cần tính **một lần** → memo theo lần gọi: `O(V + E)` thay vì hàm mũ.

**Insight 3 (háo hức + tuyến tính).** Vì giá trị tuyến tính, nếu `A` tăng `δ` thì ô `X` với `X = ... + k·A + ...` tăng `k·δ`. Lan truyền **chênh lệch** dọc cạnh ngược (Cách 3), hoặc **tính lại** các ô bị ảnh hưởng theo **thứ tự tô-pô** để mỗi ô chỉ tính một lần (Cách 4).

**Invariant (cách háo hức):** sau mỗi thao tác, `vals[r][c]` luôn là **giá trị đúng hiện tại** của mọi ô, và `dependents` phản ánh đúng các công thức đang tồn tại.

**Quy tắc nếu…thì…:**
- Nếu `set` một ô đang có công thức thì **xóa** công thức và **xóa cạnh ngược** tương ứng trước.
- Nếu `sum` ghi đè công thức cũ thì cũng xóa cạnh ngược cũ rồi mới thêm cạnh mới.
- Nếu giá trị một ô đổi thì mọi ô **đạt tới được** theo cạnh ngược phải được cập nhật.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | `set` | `get` | `sum` |
|---|---|---|---|---|
| 1 | Lười, không nhớ (mục 5) | `O(1)` | hàm mũ xấu nhất | dựng + `get` |
| 2 | Lười, memo mỗi lần `get` | `O(1)` | `O(V + E)` | dựng + `get` |
| 3 | Háo hức, lan truyền delta (DFS) | số đường đi (xấu nhất hàm mũ) | `O(1)` | dựng + lan truyền |
| 4 | Háo hức, tính lại theo tô-pô | `O(V + E)` | `O(1)` | dựng + `O(V + E)` |

(`V` = số ô ≤ 26·26 = 676, `E` = tổng số cạnh công thức.)

### 7.1. Cách 2 — Lười có nhớ theo từng lần `get`

**Giải thích.** Giống Cách 1, nhưng `_value` nhận thêm dict `memo` dùng trong **một** lần `get`. Không giữ memo giữa các lần gọi (vì `set` có thể làm nó lỗi thời) — đơn giản và đúng.

In [11]:
class ExcelLazyMemo(ExcelLazy):
    def _value_memo(self, r, c, memo):
        key = (r, c)
        if key in memo:
            return memo[key]
        f = self.formulas.get(key)
        if f is None:
            res = self.vals[r][c]
        else:
            res = 0
            for (rr, cc), k in f.items():
                res += self._value_memo(rr, cc, memo) * k
        memo[key] = res
        return res

    def get(self, row: int, column: str) -> int:
        return self._value_memo(row, ord(column) - 65, {})

ex = ExcelLazyMemo(3, "C")
ex.set(1, "A", 2)
assert ex.sum(3, "C", ["A1", "A1:B2"]) == 4
ex.set(2, "B", 2)
assert ex.get(3, "C") == 6
for depth in (8, 16, 26):
    t = time.perf_counter()
    v = diamond(ExcelLazyMemo, depth).get(depth, "A")
    print(f"depth={depth}: get = {v}, {time.perf_counter() - t:.4f}s")

depth=8: get = 128, 0.0001s
depth=16: get = 32768, 0.0002s
depth=26: get = 33554432, 0.0005s


**Đọc code:**

- Kế thừa `ExcelLazy` → dùng lại `set`, `sum`, `_cells`; chỉ thay `get`.
- `sum` gọi `self.get` → tự động dùng bản có memo.
- `memo = {}` mới cho mỗi lần `get` → không bao giờ lỗi thời.

**Code nộp LeetCode (Cách 2):** viết gọn trong một lớp `Excel` độc lập.

In [12]:
class ExcelSubmitLazy:          # trên LeetCode đổi tên thành Excel
    def __init__(self, height: int, width: str):
        self.vals = [[0] * 26 for _ in range(height + 1)]
        self.formulas = {}

    def _parse(self, numbers):
        cnt = Counter()
        for ref in numbers:
            parts = ref.split(':')
            r1, c1 = int(parts[0][1:]), ord(parts[0][0]) - 65
            r2, c2 = (int(parts[-1][1:]), ord(parts[-1][0]) - 65)
            for r in range(r1, r2 + 1):
                for c in range(c1, c2 + 1):
                    cnt[(r, c)] += 1
        return cnt

    def _eval(self, key, memo):
        if key in memo:
            return memo[key]
        f = self.formulas.get(key)
        if f is None:
            res = self.vals[key[0]][key[1]]
        else:
            res = 0
            for ref, k in f.items():
                res += self._eval(ref, memo) * k
        memo[key] = res
        return res

    def set(self, row: int, column: str, val: int) -> None:
        key = (row, ord(column) - 65)
        self.formulas.pop(key, None)
        self.vals[row][key[1]] = val

    def get(self, row: int, column: str) -> int:
        return self._eval((row, ord(column) - 65), {})

    def sum(self, row: int, column: str, numbers: List[str]) -> int:
        self.formulas[(row, ord(column) - 65)] = self._parse(numbers)
        return self.get(row, column)

ex = ExcelSubmitLazy(3, "C")
ex.set(1, "A", 2)
assert ex.sum(3, "C", ["A1", "A1:B2"]) == 4
ex.set(2, "B", 2)
assert ex.get(3, "C") == 6
print("ExcelSubmitLazy OK")

ExcelSubmitLazy OK


Mẹo trong `_parse`: `parts[-1]` là chính `parts[0]` nếu không có dấu `:` → một ô được coi như hình chữ nhật `1 × 1`. Không cần `if`.

**Độ phức tạp Cách 2:** `set` `O(1)`; `get` `O(V + E)`; `sum` `O(số ô trong vùng) + O(V + E)`. Space `O(V + E)`.

### 7.2. Cách 3 — Háo hức, lan truyền chênh lệch (DFS)

**Giải thích.** Luôn lưu **giá trị đúng** ở `vals`. Giữ `dependents[ô] = Counter(ô phụ thuộc → bội số)`.

```text
_add(ô, δ):   vals[ô] += δ;  với mỗi (dep, k) trong dependents[ô]: _add(dep, k·δ)
set(ô, v):    xóa công thức ô (gỡ cạnh ngược);  _add(ô, v - vals[ô])
sum(ô, refs): xóa công thức cũ; dựng Counter; giá trị mới = Σ vals[ref]·k;
              thêm cạnh ngược;  _add(ô, mới - vals[ô]);  trả vals[ô]
```

Tính tuyến tính đảm bảo cộng chênh lệch theo **mọi đường** cho ra đúng tổng thay đổi.

In [13]:
class ExcelDelta:
    def __init__(self, height: int, width: str):
        self.vals = [[0] * 26 for _ in range(height + 1)]
        self.formulas = {}                          # ô -> Counter(ref -> k)
        self.dependents = defaultdict(Counter)      # ref -> Counter(ô -> k)

    def _clear_formula(self, key):
        f = self.formulas.pop(key, None)
        if f:
            for ref, k in f.items():
                self.dependents[ref][key] -= k
                if self.dependents[ref][key] == 0:
                    del self.dependents[ref][key]

    def _add(self, key, delta):
        if delta == 0:
            return
        self.vals[key[0]][key[1]] += delta
        for dep, k in self.dependents[key].items():
            self._add(dep, delta * k)

    def set(self, row: int, column: str, val: int) -> None:
        key = (row, ord(column) - 65)
        self._clear_formula(key)
        self._add(key, val - self.vals[row][key[1]])

    def get(self, row: int, column: str) -> int:
        return self.vals[row][ord(column) - 65]

    def sum(self, row: int, column: str, numbers: List[str]) -> int:
        key = (row, ord(column) - 65)
        self._clear_formula(key)
        f = ExcelLazy._cells(numbers)
        self.formulas[key] = f
        new_val = 0
        for (r, c), k in f.items():
            new_val += self.vals[r][c] * k
            self.dependents[(r, c)][key] += k
        self._add(key, new_val - self.vals[row][key[1]])
        return self.vals[row][key[1]]

ex = ExcelDelta(3, "C")
ex.set(1, "A", 2)
assert ex.sum(3, "C", ["A1", "A1:B2"]) == 4
ex.set(2, "B", 2)
assert ex.get(3, "C") == 6
print("ExcelDelta OK")

ExcelDelta OK


**Đọc code:**

- `_clear_formula`: gỡ ô khỏi `dependents` của mọi ô nó từng tham chiếu — **bắt buộc** khi ghi đè (mục 4.3).
- `_add(key, delta)`: cộng chênh lệch rồi đệ quy sang các ô phụ thuộc với `delta · k`.
- `if delta == 0: return` — cắt tỉa: không đổi thì khỏi lan truyền.
- Trong `sum`, giá trị mới tính từ `vals` hiện tại (đã đúng theo invariant). Thêm cạnh ngược **trước** `_add` cũng an toàn vì không có chu trình (ô không thể tự phụ thuộc chính nó).

**Nhược điểm:** `_add` đi theo **mọi đường**, nên với "chuỗi kim cương" vẫn tốn `2^depth` lần (giống Cách 1 nhưng chuyển chi phí từ `get` sang `set`).

**Code nộp LeetCode (Cách 3):** chính lớp `ExcelDelta` (đổi tên `Excel`, thay `ExcelLazy._cells` bằng hàm parse riêng). Với ≤ 100 lời gọi trên LeetCode thường vẫn qua, nhưng không có đảm bảo tốt về lý thuyết.

In [14]:
for depth in (8, 12, 16):
    ex = diamond(ExcelDelta, depth)
    t = time.perf_counter()
    ex.set(1, "A", 2)                    # lan truyền theo mọi đường
    print(f"depth={depth}: set mất {time.perf_counter() - t:.4f}s, get = {ex.get(depth, 'A')}")

depth=8: set mất 0.0000s, get = 192
depth=12: set mất 0.0007s, get = 3072
depth=16: set mất 0.0102s, get = 49152


**Độ phức tạp Cách 3:** `get` `O(1)`; `set`/`sum` tỉ lệ với **số đường đi** trong DAG từ ô bị đổi (xấu nhất hàm mũ). Space `O(V + E)`.

### 7.3. Cách 4 — Háo hức, tính lại theo thứ tự tô-pô

**Giải thích.** Khi ô `X` đổi:

1. Tìm tập `A` các ô **bị ảnh hưởng**: mọi ô đạt tới được từ `X` theo `dependents` (DFS/BFS) — mỗi ô thăm một lần.
2. Đếm bậc vào **trong tập `A`**: với mỗi `u ∈ A ∪ {X}` và `dep ∈ dependents[u]`, `indeg[dep] += 1`.
3. Kahn: bắt đầu từ `X`, khi lấy ra một ô thì **tính lại** giá trị của nó từ công thức (trừ `X` đã có giá trị mới), giảm bậc vào các ô phụ thuộc.

Mỗi ô bị ảnh hưởng được tính lại **đúng một lần**, sau khi mọi ô nó phụ thuộc đã đúng → `O(V + E)`.

In [15]:
class ExcelTopo:
    def __init__(self, height: int, width: str):
        self.vals = [[0] * 26 for _ in range(height + 1)]
        self.formulas = {}
        self.dependents = defaultdict(set)          # ref -> tập ô phụ thuộc (bội số nằm trong formulas)

    def _clear_formula(self, key):
        f = self.formulas.pop(key, None)
        if f:
            for ref in f:
                self.dependents[ref].discard(key)

    def _recompute(self, key):
        total = 0
        for (r, c), k in self.formulas[key].items():
            total += self.vals[r][c] * k
        self.vals[key[0]][key[1]] = total

    def _propagate(self, start):
        # 1) tập bị ảnh hưởng
        affected, stack = {start}, [start]
        while stack:
            u = stack.pop()
            for v in self.dependents[u]:
                if v not in affected:
                    affected.add(v)
                    stack.append(v)
        # 2) bậc vào trong tập
        indeg = {u: 0 for u in affected}
        for u in affected:
            for v in self.dependents[u]:
                indeg[v] += 1
        # 3) Kahn + tính lại
        q = deque([start])
        while q:
            u = q.popleft()
            if u != start:
                self._recompute(u)
            for v in self.dependents[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)

    def set(self, row: int, column: str, val: int) -> None:
        key = (row, ord(column) - 65)
        self._clear_formula(key)
        self.vals[row][key[1]] = val
        self._propagate(key)

    def get(self, row: int, column: str) -> int:
        return self.vals[row][ord(column) - 65]

    def sum(self, row: int, column: str, numbers: List[str]) -> int:
        key = (row, ord(column) - 65)
        self._clear_formula(key)
        f = ExcelLazy._cells(numbers)
        self.formulas[key] = f
        for ref in f:
            self.dependents[ref].add(key)
        self._recompute(key)
        self._propagate(key)
        return self.vals[row][key[1]]

ex = ExcelTopo(3, "C")
ex.set(1, "A", 2)
assert ex.sum(3, "C", ["A1", "A1:B2"]) == 4
ex.set(2, "B", 2)
assert ex.get(3, "C") == 6
for depth in (8, 16, 26):
    ex = diamond(ExcelTopo, depth)
    t = time.perf_counter()
    ex.set(1, "A", 2)
    print(f"depth={depth}: set mất {time.perf_counter() - t:.4f}s, get = {ex.get(depth, 'A')}")

depth=8: set mất 0.0000s, get = 192
depth=16: set mất 0.0000s, get = 49152
depth=26: set mất 0.0001s, get = 50331648


**Đọc code:**

- `dependents` giờ là `set` (chỉ cần biết **ai** phụ thuộc); bội số `k` lấy từ `formulas` khi tính lại.
- Bước 2 đếm bậc vào **chỉ trong tập bị ảnh hưởng** — ô ngoài tập không đổi nên không cần chờ.
- `start` không tính lại (giá trị mới đã được đặt bởi `set` hoặc `_recompute` trong `sum`).
- Mỗi ô trong `affected` vào hàng đợi đúng một lần (khi bậc vào về 0).

**Code nộp LeetCode (Cách 4)** — xem mục 9 (lời giải chính).

**Độ phức tạp Cách 4:** `get` `O(1)`; `set` `O(V + E)`; `sum` `O(số ô trong vùng) + O(V + E)`. Space `O(V + E)`.

## 8. TRACE từng bước

Trace ví dụ chuỗi nhiều tầng ở mục 4.2 bằng Cách 4, in **bảng tính** sau mỗi thao tác và **thứ tự tô-pô** các ô được tính lại.

| Thao tác | A1 | B1 | C1 | A2 | Ô được tính lại (theo thứ tự) |
|---|---|---|---|---|---|
| `set(1,"A",5)` | 5 | 0 | 0 | 0 | — |
| `sum(1,"B",["A1","A1"])` | 5 | 10 | 0 | 0 | B1 |
| `sum(1,"C",["A1:B1"])` | 5 | 10 | 15 | 0 | C1 |
| `sum(2,"A",["B1","C1"])` | 5 | 10 | 15 | 25 | A2 |
| `set(1,"A",1)` | 1 | 2 | 3 | 5 | B1, C1, A2 |

In [16]:
def name(key):
    return f"{chr(65 + key[1])}{key[0]}"

class ExcelTopoTrace(ExcelTopo):
    def _propagate(self, start):
        affected, stack = {start}, [start]
        while stack:
            u = stack.pop()
            for v in self.dependents[u]:
                if v not in affected:
                    affected.add(v)
                    stack.append(v)
        indeg = {u: 0 for u in affected}
        for u in affected:
            for v in self.dependents[u]:
                indeg[v] += 1
        print(f"    bị ảnh hưởng: {sorted(name(x) for x in affected)}, bậc vào: { {name(k): v for k, v in indeg.items()} }")
        q, order = deque([start]), []
        while q:
            u = q.popleft()
            if u != start:
                self._recompute(u)
                order.append(f"{name(u)}={self.vals[u[0]][u[1]]}")
            for v in self.dependents[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        print(f"    tính lại theo thứ tự: {order if order else '—'}")

def show_sheet(ex, height, width):
    cols = [chr(c) for c in range(65, ord(width) + 1)]
    print("       " + "  ".join(f"{c:>4}" for c in cols))
    for r in range(1, height + 1):
        print(f"    {r:>2} " + "  ".join(f"{ex.get(r, c):>4}" for c in cols))

def trace_ops(ops, height, width, cls=ExcelTopoTrace):
    ex = cls(height, width)
    outputs = []
    for op in ops:
        meth, args = op[0], op[1:]
        res = getattr(ex, meth)(*args)
        outputs.append(res)
        print(f"{meth}{args} -> {res}")
        show_sheet(ex, height, width)
    return outputs

out = trace_ops([
    ("set", 1, "A", 5),
    ("sum", 1, "B", ["A1", "A1"]),
    ("sum", 1, "C", ["A1:B1"]),
    ("sum", 2, "A", ["B1", "C1"]),
    ("set", 1, "A", 1),
], 2, "C")
assert out == [None, 10, 15, 25, None]

    bị ảnh hưởng: ['A1'], bậc vào: {'A1': 0}
    tính lại theo thứ tự: —
set(1, 'A', 5) -> None
          A     B     C
     1    5     0     0
     2    0     0     0
    bị ảnh hưởng: ['B1'], bậc vào: {'B1': 0}
    tính lại theo thứ tự: —
sum(1, 'B', ['A1', 'A1']) -> 10
          A     B     C
     1    5    10     0
     2    0     0     0
    bị ảnh hưởng: ['C1'], bậc vào: {'C1': 0}
    tính lại theo thứ tự: —
sum(1, 'C', ['A1:B1']) -> 15
          A     B     C
     1    5    10    15
     2    0     0     0
    bị ảnh hưởng: ['A2'], bậc vào: {'A2': 0}
    tính lại theo thứ tự: —
sum(2, 'A', ['B1', 'C1']) -> 25
          A     B     C
     1    5    10    15
     2   25     0     0
    bị ảnh hưởng: ['A1', 'A2', 'B1', 'C1'], bậc vào: {'A1': 0, 'B1': 1, 'C1': 2, 'A2': 2}
    tính lại theo thứ tự: ['B1=2', 'C1=3', 'A2=5']
set(1, 'A', 1) -> None
          A     B     C
     1    1     2     3
     2    5     0     0


Trace ví dụ của đề:

In [17]:
out = trace_ops([
    ("set", 1, "A", 2),
    ("sum", 3, "C", ["A1", "A1:B2"]),
    ("set", 2, "B", 2),
    ("get", 3, "C"),
], 3, "C")
assert out == [None, 4, None, 6]

    bị ảnh hưởng: ['A1'], bậc vào: {'A1': 0}
    tính lại theo thứ tự: —
set(1, 'A', 2) -> None
          A     B     C
     1    2     0     0
     2    0     0     0
     3    0     0     0
    bị ảnh hưởng: ['C3'], bậc vào: {'C3': 0}
    tính lại theo thứ tự: —
sum(3, 'C', ['A1', 'A1:B2']) -> 4
          A     B     C
     1    2     0     0
     2    0     0     0
     3    0     0     4
    bị ảnh hưởng: ['B2', 'C3'], bậc vào: {'C3': 1, 'B2': 0}
    tính lại theo thứ tự: ['C3=6']
set(2, 'B', 2) -> None
          A     B     C
     1    2     0     0
     2    0     2     0
     3    0     0     6
get(3, 'C') -> 6
          A     B     C
     1    2     0     0
     2    0     2     0
     3    0     0     6


## 9. Code nộp LeetCode

Lời giải chính: **Háo hức + tính lại theo thứ tự tô-pô** (Cách 4) — `get` `O(1)`, `set`/`sum` `O(V + E)` có đảm bảo.

In [18]:
from collections import Counter, defaultdict, deque
from typing import List

class Excel:
    def __init__(self, height: int, width: str):
        self.vals = [[0] * 26 for _ in range(height + 1)]   # hàng 1..height, cột 0..25
        self.formulas = {}                                  # ô -> Counter(ô tham chiếu -> bội số)
        self.dependents = defaultdict(set)                  # ô -> tập ô có công thức dùng nó

    # ---------- tiện ích ----------
    @staticmethod
    def _parse(numbers: List[str]) -> Counter:
        cnt = Counter()
        for ref in numbers:
            parts = ref.split(':')
            r1, c1 = int(parts[0][1:]), ord(parts[0][0]) - 65
            r2, c2 = int(parts[-1][1:]), ord(parts[-1][0]) - 65
            for r in range(r1, r2 + 1):
                for c in range(c1, c2 + 1):
                    cnt[(r, c)] += 1
        return cnt

    def _clear_formula(self, key) -> None:
        f = self.formulas.pop(key, None)
        if f:
            for ref in f:
                self.dependents[ref].discard(key)

    def _recompute(self, key) -> None:
        total = 0
        for (r, c), k in self.formulas[key].items():
            total += self.vals[r][c] * k
        self.vals[key[0]][key[1]] = total

    def _propagate(self, start) -> None:
        affected, stack = {start}, [start]
        while stack:
            u = stack.pop()
            for v in self.dependents[u]:
                if v not in affected:
                    affected.add(v)
                    stack.append(v)
        indeg = dict.fromkeys(affected, 0)
        for u in affected:
            for v in self.dependents[u]:
                indeg[v] += 1
        q = deque([start])
        while q:
            u = q.popleft()
            if u != start:
                self._recompute(u)
            for v in self.dependents[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)

    # ---------- API ----------
    def set(self, row: int, column: str, val: int) -> None:
        key = (row, ord(column) - 65)
        self._clear_formula(key)
        self.vals[row][key[1]] = val
        self._propagate(key)

    def get(self, row: int, column: str) -> int:
        return self.vals[row][ord(column) - 65]

    def sum(self, row: int, column: str, numbers: List[str]) -> int:
        key = (row, ord(column) - 65)
        self._clear_formula(key)
        f = self._parse(numbers)
        self.formulas[key] = f
        for ref in f:
            self.dependents[ref].add(key)
        self._recompute(key)
        self._propagate(key)
        return self.vals[row][key[1]]

ex = Excel(3, "C")
ex.set(1, "A", 2)
assert ex.sum(3, "C", ["A1", "A1:B2"]) == 4
ex.set(2, "B", 2)
assert ex.get(3, "C") == 6
print("Excel OK")

Excel OK


## 10. Vì sao thuật toán đúng

**Định nghĩa.** Giá trị đúng `val*(X)` của ô `X`: nếu `X` là ô số thì là số đó; nếu là ô công thức thì `Σ k · val*(ref)`. Vì không có chu trình, định nghĩa đệ quy này xác định duy nhất.

**Cách 1/2 (lười):** `_value` cài đặt **đúng nguyên văn** định nghĩa trên. Memo chỉ tránh tính lại cùng một ô trong một lần gọi — trong một lần `get` không có thay đổi nào, nên giá trị đã nhớ vẫn đúng.

**Cách 4 (tô-pô) — invariant:** trước và sau mỗi thao tác, `vals[X] = val*(X)` với mọi `X`.

- Thao tác chỉ thay đổi định nghĩa của **một** ô `S` (số mới hoặc công thức mới). Ô `Y` có `val*` có thể đổi **chỉ khi** `Y` đạt tới được từ `S` theo cạnh phụ thuộc (tập `affected`). Ô ngoài tập giữ nguyên giá trị đúng.
- `S` được gán giá trị đúng trước (số mới, hoặc `_recompute` từ các ô nó tham chiếu — các ô này không thuộc `affected` vì không có chu trình, nên đang đúng).
- Kahn xử lý `affected` theo thứ tự tô-pô: khi `Y` được lấy ra, mọi ô `Y` tham chiếu **nằm trong `affected`** đã được xử lý (bậc vào về 0), còn ô ngoài `affected` vốn đúng → `_recompute(Y)` cho `val*(Y)`.

**Cách 3 (delta):** vì `val*` tuyến tính theo giá trị các ô lá, thay đổi `δ` ở `S` làm `Y` đổi `δ · (tổng tích bội số trên mọi đường S → Y)`; DFS `_add` cộng đúng từng đường một lần.

**Xóa cạnh khi ghi đè:** đảm bảo `dependents` chỉ chứa công thức đang tồn tại; nếu không, ô đã bị `set` thành số vẫn bị "cập nhật" sai.

## 11. Phân tích Time/Space

| Cách | `set` | `get` | `sum` | Space |
|---|---|---|---|---|
| 1. Lười | `O(1)` | hàm mũ xấu nhất | `O(R)` + `get` | `O(V + E)` |
| 2. Lười + memo | `O(1)` | `O(V + E)` | `O(R + V + E)` | `O(V + E)` |
| 3. Delta | số đường đi | `O(1)` | `O(R)` + lan truyền | `O(V + E)` |
| 4. Tô-pô | `O(V + E)` | `O(1)` | `O(R + V + E)` | `O(V + E)` |

`R` = tổng số ô trong các vùng của `numbers` (≤ 5 · 676). Với `V <= 676` và ≤ 100 lời gọi, Cách 2 và 4 đều rất nhanh. **Chọn cách nào?** Nếu `get` nhiều hơn hẳn `set` → háo hức (Cách 4). Nếu `set` nhiều → lười (Cách 2).

In [19]:
import random
random.seed(1)
def heavy_ops(n_ops=100, H=26, W="Z"):
    ops = []
    for _ in range(n_ops):
        r, c = random.randint(1, H), chr(random.randint(65, ord(W)))
        if random.random() < 0.3:
            ops.append(("set", r, c, random.randint(-100, 100)))
        elif random.random() < 0.5:
            ops.append(("get", r, c))
        else:
            # chỉ tham chiếu các hàng phía trên => không có chu trình
            if r == 1:
                ops.append(("set", r, c, 1))
                continue
            refs = []
            for _ in range(random.randint(1, 5)):
                r1 = random.randint(1, r - 1); r2 = random.randint(r1, r - 1)
                c1 = random.randint(65, ord(W)); c2 = random.randint(c1, ord(W))
                refs.append(f"{chr(c1)}{r1}:{chr(c2)}{r2}" if random.random() < 0.5 else f"{chr(c1)}{r1}")
            ops.append(("sum", r, c, refs))
    return ops

ops = heavy_ops()
for cls in (ExcelLazyMemo, ExcelTopo, Excel):
    t = time.perf_counter()
    ex = cls(26, "Z")
    res = [getattr(ex, op[0])(*op[1:]) for op in ops]
    print(f"{cls.__name__:>14}: {time.perf_counter() - t:.3f}s, checksum={sum(x for x in res if x is not None)}")

 ExcelLazyMemo: 0.002s, checksum=1880
     ExcelTopo: 0.002s, checksum=1880
         Excel: 0.002s, checksum=1880


## 12. Bộ test

Gồm: ví dụ đề, edge case (ghi đè công thức bằng `set`, ghi đè bằng `sum` khác, ô tham chiếu trùng nhiều lần, vùng 1×1, chuỗi nhiều tầng) và **thao tác ngẫu nhiên có seed** — sau mỗi thao tác so **toàn bộ bảng** giữa các cách với Cách 1 (trọng tài).

In [20]:
CLASSES = [ExcelLazy, ExcelLazyMemo, ExcelSubmitLazy, ExcelDelta, ExcelTopo, Excel]

def run_all(ops, H, W):
    objs = [cls(H, W) for cls in CLASSES]
    for op in ops:
        outs = [getattr(o, op[0])(*op[1:]) for o in objs]
        assert len(set(map(str, outs))) == 1, (op, outs)
        for r in range(1, H + 1):
            for c in range(65, ord(W) + 1):
                vals = [o.get(r, chr(c)) for o in objs]
                assert len(set(vals)) == 1, (op, r, chr(c), vals)
    return outs

# Ví dụ đề
assert run_all([("set", 1, "A", 2), ("sum", 3, "C", ["A1", "A1:B2"]), ("set", 2, "B", 2), ("get", 3, "C")], 3, "C") == [6] * len(CLASSES)

# Ghi đè công thức bằng set: đổi A1 không còn ảnh hưởng B1
ex = Excel(2, "B")
ex.set(1, "A", 3); ex.sum(1, "B", ["A1"]); ex.set(1, "B", 7); ex.set(1, "A", 100)
assert ex.get(1, "B") == 7

# Ghi đè công thức bằng sum khác
ex = Excel(2, "B")
ex.set(1, "A", 3); ex.set(2, "A", 5)
assert ex.sum(1, "B", ["A1"]) == 3
assert ex.sum(1, "B", ["A2"]) == 5
ex.set(1, "A", 100)
assert ex.get(1, "B") == 5

# Tham chiếu trùng nhiều lần + vùng 1x1
ex = Excel(1, "B")
ex.set(1, "A", 4)
assert ex.sum(1, "B", ["A1", "A1:A1", "A1"]) == 12

# Chuỗi nhiều tầng (mục 4.2)
assert run_all([("set", 1, "A", 5), ("sum", 1, "B", ["A1", "A1"]), ("sum", 1, "C", ["A1:B1"]),
                ("sum", 2, "A", ["B1", "C1"]), ("set", 1, "A", 1), ("get", 2, "A")], 2, "C") == [5] * len(CLASSES)

# Ô công thức tham chiếu ô công thức rồi ô giữa bị set thành số
ex = Excel(3, "A")
ex.set(1, "A", 1); ex.sum(2, "A", ["A1"]); ex.sum(3, "A", ["A2", "A2"])
ex.set(2, "A", 10)
assert ex.get(3, "A") == 20
ex.set(1, "A", 50)
assert ex.get(3, "A") == 20
print("Test cố định OK")

Test cố định OK


In [21]:
def depends_on(objs_formulas, a, b):
    """True nếu ô a (trực tiếp/gián tiếp) phụ thuộc ô b, dựa trên dict formulas."""
    stack, seen = [a], set()
    while stack:
        u = stack.pop()
        if u == b:
            return True
        if u in seen:
            continue
        seen.add(u)
        stack.extend(objs_formulas.get(u, {}).keys())
    return False

def random_ops(n_ops, H, W, rng):
    ref_model = ExcelLazy(H, W)                    # dùng để tránh tạo chu trình
    ops = []
    while len(ops) < n_ops:
        r, c = rng.randint(1, H), chr(rng.randint(65, ord(W)))
        x = rng.random()
        if x < 0.35:
            op = ("set", r, c, rng.randint(-100, 100))
        elif x < 0.55:
            op = ("get", r, c)
        else:
            refs = []
            for _ in range(rng.randint(1, 3)):
                r1 = rng.randint(1, H); r2 = rng.randint(r1, H)
                c1 = rng.randint(65, ord(W)); c2 = rng.randint(c1, ord(W))
                refs.append(f"{chr(c1)}{r1}:{chr(c2)}{r2}" if rng.random() < 0.5 else f"{chr(c1)}{r1}")
            target = (r, ord(c) - 65)
            cells_ref = ExcelLazy._cells(refs)
            if any(ref == target or depends_on(ref_model.formulas, ref, target) for ref in cells_ref):
                continue                           # sẽ tạo chu trình -> bỏ
            op = ("sum", r, c, refs)
        getattr(ref_model, op[0])(*op[1:])
        ops.append(op)
    return ops

rng = random.Random(2024)
for trial in range(60):
    H, W = rng.randint(1, 4), chr(rng.randint(65, 68))
    run_all(random_ops(40, H, W, rng), H, W)
print("60 chuỗi thao tác ngẫu nhiên: mọi cách khớp nhau trên toàn bảng sau từng bước")

60 chuỗi thao tác ngẫu nhiên: mọi cách khớp nhau trên toàn bảng sau từng bước


## 13. Lỗi người mới hay gặp

### Lỗi 1: Lưu **giá trị** tổng thay vì **công thức**

**Code sai:** `sum` tính tổng một lần rồi `set` ô bằng số đó.

**Vì sao sai:** đề yêu cầu công thức tồn tại lâu dài — đổi ô tham chiếu phải cập nhật ô tổng.

In [22]:
class ExcelSnapshotWrong(ExcelLazy):
    def sum(self, row, column, numbers):
        total = 0
        for (r, c), k in self._cells(numbers).items():
            total += self._value(r, c) * k
        self.set(row, column, total)          # SAI: chỉ lưu con số
        return total

for cls in (ExcelSnapshotWrong, Excel):
    ex = cls(3, "C"); ex.set(1, "A", 2); ex.sum(3, "C", ["A1", "A1:B2"]); ex.set(2, "B", 2)
    print(f"{cls.__name__:>18}: get(3,'C') = {ex.get(3, 'C')}")

ExcelSnapshotWrong: get(3,'C') = 4
             Excel: get(3,'C') = 6


**Code đúng:** lưu `Counter` tham chiếu (công thức) và tính lại khi cần.

### Lỗi 2: Dùng `set` thay vì `Counter` → mất bội số

**Code sai:** `formula = set(cells)` — `["A1", "A1:B2"]` chỉ cộng `A1` một lần.

In [23]:
class ExcelSetWrong(ExcelLazy):
    @staticmethod
    def _cells(numbers):
        s = set()
        for ref in numbers:
            for r, col in expand(ref):
                s.add((r, ord(col) - 65))
        return Counter(s)                      # SAI: mỗi ô bội số 1

for cls in (ExcelSetWrong, Excel):
    ex = cls(3, "C"); ex.set(1, "A", 2)
    print(f"{cls.__name__:>14}: sum = {ex.sum(3, 'C', ['A1', 'A1:B2'])}")

 ExcelSetWrong: sum = 2
         Excel: sum = 4


**Code đúng:** `Counter` cộng dồn mọi lần xuất hiện.

### Lỗi 3: Cách háo hức quên gỡ cạnh ngược khi ghi đè

**Code sai:** `set` chỉ xóa `formulas[key]` mà không xóa `key` khỏi `dependents[ref]`.

**Vì sao sai:** sau đó đổi `ref` vẫn "lan truyền" vào ô đã thành số.

In [24]:
class ExcelDeltaNoClearWrong(ExcelDelta):
    def _clear_formula(self, key):
        self.formulas.pop(key, None)           # SAI: không gỡ dependents

for cls in (ExcelDeltaNoClearWrong, Excel):
    ex = cls(2, "B")
    ex.set(1, "A", 3); ex.sum(1, "B", ["A1"]); ex.set(1, "B", 7); ex.set(1, "A", 100)
    print(f"{cls.__name__:>22}: B1 = {ex.get(1, 'B')} (đúng phải là 7)")

ExcelDeltaNoClearWrong: B1 = 104 (đúng phải là 7)
                 Excel: B1 = 7 (đúng phải là 7)


**Code đúng:** `_clear_formula` duyệt công thức cũ và gỡ từng cạnh ngược.

### Lỗi 4: Phân tích hàng chỉ lấy một chữ số

**Code sai:** `row = int(ref[1])` — `"A12"` bị đọc thành hàng 1.

In [25]:
def parse_wrong(ref):
    return int(ref[1]), ref[0]                 # SAI với hàng >= 10
print("sai :", parse_wrong("A12"), "| đúng:", parse_cell("A12"))

sai : (1, 'A') | đúng: (12, 'A')


**Code đúng:** `int(ref[1:])`.

### Lỗi 5: Cách háo hức cập nhật dependents theo thứ tự BFS thường (không tô-pô)

Nếu duyệt BFS từ ô bị đổi và tính lại mỗi ô **ngay khi gặp lần đầu**, một ô có thể được tính trước khi một ô nó phụ thuộc (ở đường dài hơn) được cập nhật → giá trị sai. Phải dùng thứ tự tô-pô (Cách 4) hoặc delta (Cách 3).

In [26]:
class ExcelBfsWrong(ExcelTopo):
    def _propagate(self, start):
        seen, q = {start}, deque([start])
        while q:
            u = q.popleft()
            for v in self.dependents[u]:
                if v not in seen:
                    seen.add(v)
                    self._recompute(v)         # SAI: có thể tính trước khi phụ thuộc của v cập nhật
                    q.append(v)

ops_demo = [("set", 1, "A", 5), ("sum", 1, "B", ["A1"]), ("sum", 1, "C", ["B1"]),
            ("sum", 2, "A", ["A1", "C1"]), ("set", 1, "A", 1), ("get", 2, "A")]
for cls in (ExcelBfsWrong, Excel):
    ex = cls(2, "C")
    print(f"{cls.__name__:>14}: A2 = {[getattr(ex, op[0])(*op[1:]) for op in ops_demo][-1]} (đúng: 1 + 1 = 2)")

 ExcelBfsWrong: A2 = 6 (đúng: 1 + 1 = 2)
         Excel: A2 = 2 (đúng: 1 + 1 = 2)


## 14. Mẹo & nhận dạng pattern

- **Bài thiết kế có "giá trị phụ thuộc giá trị khác"** → mô hình hóa thành đồ thị phụ thuộc; hỏi ngay: có chu trình không? Lười hay háo hức?
- **Lười + memo theo lần gọi** là lời giải ngắn, an toàn, đủ nhanh khi số thao tác nhỏ.
- **Háo hức** cần: cạnh ngược, gỡ cạnh khi ghi đè, cập nhật theo **thứ tự tô-pô**.
- Giá trị **tuyến tính** theo đầu vào → có thể lan truyền **chênh lệch**.
- Bội số tham chiếu → `Counter`, không phải `set`.
- Liên quan: Spreadsheet (bảng tính) thực tế, build system (Make: chỉ build lại phần phụ thuộc), LC 2296/2307, LC 207/210 (Course Schedule — sắp xếp tô-pô).

## 15. Biến thể

1. **Phát hiện chu trình** — trước khi lưu công thức, kiểm tra ô mục tiêu có nằm trong tập đạt tới được từ các ô tham chiếu (theo `formulas`) không; nếu có, từ chối.
2. **Công thức khác ngoài tổng** (`max`, `min`, nhân) — cách lười vẫn dùng được; cách delta **không** dùng được (không tuyến tính); cách tô-pô vẫn dùng được.
3. **Undo** — lưu lịch sử thao tác, hoặc lưu trạng thái cũ của ô bị ghi đè.
4. **Bảng lớn, thưa** — dùng dict thay vì mảng 2D.

In [27]:
class ExcelMax(ExcelTopo):
    """Biến thể 2: công thức MAX thay vì SUM (bội số không còn ý nghĩa)."""
    def _recompute(self, key):
        self.vals[key[0]][key[1]] = max(self.vals[r][c] for (r, c) in self.formulas[key])

ex = ExcelMax(2, "C")
ex.set(1, "A", 3); ex.set(1, "B", 9)
print("max(A1:B1) =", ex.sum(1, "C", ["A1:B1"]))
ex.set(1, "A", 20)
print("sau set A1 = 20:", ex.get(1, "C"))
assert ex.get(1, "C") == 20

def would_cycle(excel_obj, row, column, numbers):
    """Biến thể 1: True nếu đặt công thức này tạo chu trình."""
    target = (row, ord(column) - 65)
    for ref in Excel._parse(numbers):
        if ref == target or depends_on(excel_obj.formulas, ref, target):
            return True
    return False

ex = Excel(2, "B")
ex.sum(1, "B", ["A1"])
print("B1 = A1; đặt A1 = B1 có tạo chu trình?", would_cycle(ex, 1, "A", ["B1"]))
print("đặt A2 = B1 có tạo chu trình?", would_cycle(ex, 2, "A", ["B1"]))

max(A1:B1) = 9
sau set A1 = 20: 20
B1 = A1; đặt A1 = B1 có tạo chu trình? True
đặt A2 = B1 có tạo chu trình? False


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | `set` | `get` | Khi nào dùng |
|---|---|---|---|---|
| 1. Lười | đệ quy theo định nghĩa | `O(1)` | hàm mũ xấu nhất | trọng tài, phụ thuộc nông |
| 2. Lười + memo | nhớ trong một lần `get` | `O(1)` | `O(V + E)` | ngắn, an toàn, nhiều `set` |
| 3. Delta | lan truyền `k·δ` theo cạnh ngược | số đường đi | `O(1)` | công thức tuyến tính, DAG thưa |
| 4. Tô-pô | tính lại tập bị ảnh hưởng theo Kahn | `O(V + E)` | `O(1)` | nhiều `get`, công thức bất kỳ |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code:

```text
e = Excel(2, "B"); e.set(1,"A",1); e.set(1,"B",2)
e.sum(2,"A",["A1:B1","B1"])  -> ?
e.set(1,"B",5); e.get(2,"A") -> ?
```

**Bài 2 (tính tay).** Với chuỗi thao tác ở mục 4.2, sau `set(1,"A",1)` hãy viết thứ tự tô-pô các ô được tính lại và giá trị từng ô.

**Bài 3 (điền TODO).** Hoàn thiện `parse_refs(numbers)` trả về `Counter` các ô `(hàng, chỉ số cột)`.

**Bài 4 (tự viết lại).** Viết lại lớp lười có memo `MyExcel` (Cách 2) mà không nhìn code.

**Bài 5 (mini challenge).** Thêm phương thức `clear(row, column)` vào lớp `Excel` (Cách 4): đặt ô về 0 và xóa công thức (nếu có), cập nhật các ô phụ thuộc. Kiểm tra bằng thao tác ngẫu nhiên so với cách lười.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def parse_refs(numbers):
    # TODO: split(':'), int(ref[1:]), ord(ref[0]) - 65, Counter
    pass

# Bài 4 — TODO
class MyExcel:
    def __init__(self, height, width):
        # TODO: vals, formulas
        pass
    def set(self, row, column, val):
        pass
    def get(self, row, column):
        pass
    def sum(self, row, column, numbers):
        pass

# Bài 5 — TODO
def add_clear_method(cls):
    # TODO: định nghĩa clear(self, row, column) gọi lại set(row, column, 0)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `"A1:B1"` là `A1 + B1`, cộng thêm `B1` nữa.
- Mức 2: `1 + 2 + 2 = 5`.
- Mức 3: sau `B1 = 5`: `1 + 5 + 5 = 11`.

**Bài 2.**
- Mức 1: tập bị ảnh hưởng của `A1`: `B1, C1, A2`.
- Mức 2: `B1` chỉ phụ thuộc `A1`; `C1` phụ thuộc `A1, B1`; `A2` phụ thuộc `B1, C1`.
- Mức 3: thứ tự `B1 = 2`, `C1 = 3`, `A2 = 5`.

**Bài 3.**
- Mức 1: với mỗi `ref`, `parts = ref.split(':')`.
- Mức 2: góc đầu `parts[0]`, góc cuối `parts[-1]`.
- Mức 3: hai vòng `for` hàng/cột, `cnt[(r, c)] += 1`.

**Bài 4.**
- Mức 1: `vals` mảng 2D, `formulas` dict.
- Mức 2: `get` gọi hàm đệ quy có `memo = {}` mới.
- Mức 3: `set` nhớ `formulas.pop(key, None)`.

**Bài 5.**
- Mức 1: `clear` tương đương `set(row, column, 0)`.
- Mức 2: gán `cls.clear = lambda self, r, c: self.set(r, c, 0)`.
- Mức 3: thêm `("clear", r, c)` vào bộ sinh thao tác và so sánh.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
e = Excel(2, "B"); e.set(1, "A", 1); e.set(1, "B", 2)
assert e.sum(2, "A", ["A1:B1", "B1"]) == 5
e.set(1, "B", 5)
assert e.get(2, "A") == 11

# Bài 2
out = trace_ops([("set", 1, "A", 5), ("sum", 1, "B", ["A1", "A1"]), ("sum", 1, "C", ["A1:B1"]),
                 ("sum", 2, "A", ["B1", "C1"]), ("set", 1, "A", 1)], 2, "C")

# Bài 3
def parse_refs_ans(numbers):
    cnt = Counter()
    for ref in numbers:
        parts = ref.split(':')
        r1, c1 = int(parts[0][1:]), ord(parts[0][0]) - 65
        r2, c2 = int(parts[-1][1:]), ord(parts[-1][0]) - 65
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                cnt[(r, c)] += 1
    return cnt

assert parse_refs_ans(["A1", "A1:B2"]) == Excel._parse(["A1", "A1:B2"])
assert parse_refs_ans(["C12"]) == Counter({(12, 2): 1})
print("Bài 1–3 OK")

    bị ảnh hưởng: ['A1'], bậc vào: {'A1': 0}
    tính lại theo thứ tự: —
set(1, 'A', 5) -> None
          A     B     C
     1    5     0     0
     2    0     0     0
    bị ảnh hưởng: ['B1'], bậc vào: {'B1': 0}
    tính lại theo thứ tự: —
sum(1, 'B', ['A1', 'A1']) -> 10
          A     B     C
     1    5    10     0
     2    0     0     0
    bị ảnh hưởng: ['C1'], bậc vào: {'C1': 0}
    tính lại theo thứ tự: —
sum(1, 'C', ['A1:B1']) -> 15
          A     B     C
     1    5    10    15
     2    0     0     0
    bị ảnh hưởng: ['A2'], bậc vào: {'A2': 0}
    tính lại theo thứ tự: —
sum(2, 'A', ['B1', 'C1']) -> 25
          A     B     C
     1    5    10    15
     2   25     0     0
    bị ảnh hưởng: ['A1', 'A2', 'B1', 'C1'], bậc vào: {'A1': 0, 'B1': 1, 'C1': 2, 'A2': 2}
    tính lại theo thứ tự: ['B1=2', 'C1=3', 'A2=5']
set(1, 'A', 1) -> None
          A     B     C
     1    1     2     3
     2    5     0     0
Bài 1–3 OK


In [30]:
# Bài 4
class MyExcelAns:
    def __init__(self, height, width):
        self.vals = [[0] * 26 for _ in range(height + 1)]
        self.formulas = {}

    def _ev(self, key, memo):
        if key not in memo:
            f = self.formulas.get(key)
            memo[key] = self.vals[key[0]][key[1]] if f is None else \
                sum(self._ev(ref, memo) * k for ref, k in f.items())
        return memo[key]

    def set(self, row, column, val):
        key = (row, ord(column) - 65)
        self.formulas.pop(key, None)
        self.vals[row][key[1]] = val

    def get(self, row, column):
        return self._ev((row, ord(column) - 65), {})

    def sum(self, row, column, numbers):
        self.formulas[(row, ord(column) - 65)] = parse_refs_ans(numbers)
        return self.get(row, column)

# Bài 5
def add_clear_method_ans(cls):
    cls.clear = lambda self, row, column: self.set(row, column, 0)
    return cls

ExcelWithClear = add_clear_method_ans(type("ExcelWithClear", (Excel,), {}))
LazyWithClear = add_clear_method_ans(type("LazyWithClear", (MyExcelAns,), {}))

rng = random.Random(77)
for trial in range(40):
    H, W = rng.randint(1, 4), chr(rng.randint(65, 68))
    ops = random_ops(30, H, W, rng)
    ops = [("clear", op[1], op[2]) if op[0] == "set" and rng.random() < 0.3 else op for op in ops]
    a, b = ExcelWithClear(H, W), LazyWithClear(H, W)
    for op in ops:
        ra, rb = getattr(a, op[0])(*op[1:]), getattr(b, op[0])(*op[1:])
        assert ra == rb, op
        assert all(a.get(r, chr(c)) == b.get(r, chr(c)) for r in range(1, H + 1) for c in range(65, ord(W) + 1))
print("Bài 4–5 OK (đối chiếu trên thao tác ngẫu nhiên)")

Bài 4–5 OK (đối chiếu trên thao tác ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template lười + memo (ngắn nhất):**

```python
class Excel:
    def __init__(self, H, W):
        self.vals = [[0]*26 for _ in range(H+1)]; self.f = {}
    def _ev(self, key, memo):
        if key not in memo:
            f = self.f.get(key)
            memo[key] = self.vals[key[0]][key[1]] if f is None else \
                sum(self._ev(r, memo)*k for r, k in f.items())
        return memo[key]
    def set(self, r, c, v): k = (r, ord(c)-65); self.f.pop(k, None); self.vals[r][k[1]] = v
    def get(self, r, c): return self._ev((r, ord(c)-65), {})
    def sum(self, r, c, nums): self.f[(r, ord(c)-65)] = parse(nums); return self.get(r, c)
```

**Template háo hức tô-pô:** `affected` (DFS theo `dependents`) → `indeg` trong `affected` → Kahn từ ô bị đổi, `_recompute` từng ô lấy ra.

**Checklist:**

- [ ] `sum` lưu **công thức**, không lưu con số.
- [ ] Bội số tham chiếu bằng `Counter`.
- [ ] `set` xóa công thức cũ; cách háo hức gỡ cả cạnh ngược.
- [ ] Hàng có thể 2 chữ số: `int(ref[1:])`.
- [ ] Vùng `"X:Y"` duyệt đủ hình chữ nhật (bao gồm hai đầu).
- [ ] Háo hức: tính lại theo **thứ tự tô-pô**.
- [ ] Tên lớp/method đúng LeetCode: `Excel`, `set`, `get`, `sum`.